In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv


ModuleNotFoundError: No module named 'google.generativeai'

In [2]:
from google import genai
from google.genai import types

In [3]:
# 1. Load API Key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# 2. Khởi tạo Client
client = genai.Client(api_key=GOOGLE_API_KEY)

In [11]:
# 3. Định nghĩa đường dẫn
# Input file
input_file_path = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA.pdf")

# Lấy tên thư mục cha của file (ví dụ: "Ngoai_ngu")
subfolder_name = input_file_path.parent.name

# Thư mục gốc cho dữ liệu đã tiền xử lý
base_output_dir = Path(r"C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Preprocessed_Data")

# Tạo đường dẫn đầy đủ: Preprocessed_Data/Ngoai_ngu
final_output_folder = base_output_dir / subfolder_name

# Đảm bảo thư mục đích tồn tại (tạo mới nếu chưa có)
final_output_folder.mkdir(parents=True, exist_ok=True)

# Tên file đầu ra
output_file_path = final_output_folder / f"{input_file_path.stem}_preprocessed.txt"

In [ ]:
# 4. Thiết lập System Prompt
system_instruction = """
Bạn là chuyên gia tiền xử lý dữ liệu RAG cho hệ thống Quy chế đào tạo ĐHBK Hà Nội.

NHIỆM VỤ:
Phân tích tài liệu PDF, sửa lỗi trình bày, chuyển bảng sang Markdown, và chia thành các chunk logic.

THÔNG TIN TÀI LIỆU:
- Tên file: {filename}
- parent_doc_id: {doc_id}
- category: {category}
- year: {year}

CHIẾN LƯỢC CHIA CHUNK:
- Đơn vị cơ bản: mỗi Điều = 1 chunk
- Nếu Điều quá ngắn (< 80 từ): gộp với Điều liền kề cùng chủ đề
- Nếu Điều quá dài (> 600 từ): tách theo Khoản, mỗi Khoản là 1 chunk
- Bảng biểu lớn (> 5 hàng): có thể là chunk độc lập
- Phần mở đầu/định nghĩa chung: 1 chunk riêng

QUY TẮC ĐỊNH DẠNG:
1. Mỗi chunk nằm trong <<<CHUNK_START>>> ... <<<CHUNK_END>>>
2. Không trả về bất kỳ lời dẫn, giải thích hay hội thoại nào ngoài các thẻ trên
3. Bảng biểu dùng định dạng Markdown table
4. Hình ảnh/biểu đồ: mô tả bằng [Hình: <nội dung>]
5. Trả lời hoàn toàn bằng tiếng Việt

CẤU TRÚC CHUNK (tuân thủ chính xác, không thêm/bớt field):
<<<CHUNK_START>>>
source: {filename}
parent_doc_id: {doc_id}
chunk_id: {doc_id}_001  ← tăng dần, 3 chữ số, ví dụ _002, _003
chunk_index: 1          ← số nguyên tăng dần
language: vi
category: {category}
year: {year}
chunk_title: [Tên điều khoản hoặc chủ đề chính]
topic_tags: [tag1, tag2, tag3]
summary: [2-3 câu tóm tắt, ưu tiên từ khóa: mã học phần, tín chỉ, cảnh báo học tập, điều kiện xét]

[Nội dung chunk ở đây]
<<<CHUNK_END>>>

VÍ DỤ OUTPUT:
<<<CHUNK_START>>>
source: Ngoai_ngu_2022_Quy_doi_CCTA.pdf
parent_doc_id: Ngoai_ngu_2022_Quy_doi_CCTA
chunk_id: Ngoai_ngu_2022_Quy_doi_CCTA_001
chunk_index: 1
language: vi
category: Ngoai_ngu
year: 2022
chunk_title: Điều 1 – Phạm vi áp dụng
topic_tags: [chứng chỉ ngoại ngữ, quy đổi, điều kiện tốt nghiệp]
summary: Quy định về việc quy đổi chứng chỉ ngoại ngữ quốc tế sang điểm học phần tương đương tại ĐHBK Hà Nội. Áp dụng cho sinh viên đại học chính quy có chứng chỉ IELTS, TOEIC, hoặc tương đương.

Điều 1. Phạm vi áp dụng

Quy định này áp dụng cho sinh viên đại học chính quy của Đại học Bách khoa Hà Nội có nhu cầu quy đổi chứng chỉ ngoại ngữ quốc tế...
<<<CHUNK_END>>>
"""

In [23]:
def process_document():
    print(f"--- Đang upload file PDF: {input_file_path.name} ---")
    
    # Upload file lên Gemini File API
    file_upload = client.files.upload(file=input_file_path)
    
    # Chờ file được xử lý xong trên server (PDF cần thời gian OCR/Parsing)
    while file_upload.state == "PROCESSING":
        print(".", end="", flush=True)
        time.sleep(2)
        file_upload = client.files.get(name=file_upload.name)
    
    print(f"\n--- File ready! Đang gửi yêu cầu cho Gemini 3.1 Flash Lite ---")

    # Gửi yêu cầu kèm file đã upload
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite", 
        contents=[file_upload],
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.1
        )
    )
    
    # Logic loại bỏ text thừa (nếu có)
    raw_text = response.text
    if "<<<CHUNK>>>" in raw_text:
        clean_content = "<<<CHUNK>>>" + raw_text.split("<<<CHUNK>>>", 1)[-1]
    else:
        clean_content = raw_text

    # 4. Lưu kết quả
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(clean_content.strip())
    
    # Xóa file trên server sau khi xử lý xong (optional)
    client.files.delete(name=file_upload.name)
    
    print(f"--- Hoàn thành! File lưu tại: {output_file_path} ---")

In [24]:
process_document()

--- Đang upload file PDF: Ngoai_ngu_2022_Quy_doi_CCTA.pdf ---

--- File ready! Đang gửi yêu cầu cho Gemini 3.1 Flash Lite ---
--- Hoàn thành! File lưu tại: C:\Users\LENOVO\Documents\GitHub\Chatbot QCDT V2\Preprocessed_Data\Ngoai_ngu\Ngoai_ngu_2022_Quy_doi_CCTA_preprocessed.txt ---
